In [3]:
from pathlib import Path

import pandas as pd

data_dir = Path(r"C:\Users\Younis\Desktop\Data Science\archive 2\archive\TrainData\TrainData")

def read_csv(filename):
    return pd.read_csv(data_dir / filename, dtype={"CustomerID": "string"})

claims = read_csv("Train_Claim.csv")
demographics = read_csv("Train_Demographics.csv")
policies = read_csv("Train_Policy.csv")
vehicles = read_csv("Train_Vehicle.csv")
labels = read_csv("Traindata_with_Target.csv")

vehicles_wide = vehicles.pivot(
    index="CustomerID",
    columns="VehicleAttribute",
    values="VehicleAttributeDetails",
).reset_index()

data = (
    claims
    .merge(demographics, on="CustomerID", validate="one_to_one")
    .merge(policies, on="CustomerID", validate="one_to_one")
    .merge(vehicles_wide, on="CustomerID", validate="one_to_one")
    .merge(labels, on="CustomerID", validate="one_to_one")
)

print("Dataset shape:", data.shape)
print(data["ReportedFraud"].value_counts())

Dataset shape: (28836, 42)
ReportedFraud
N    21051
Y     7785
Name: count, dtype: int64


In [4]:
feature_columns = [
    "CustomerLoyaltyPeriod",
    "InsurancePolicyState",
    "Policy_CombinedSingleLimit",
    "Policy_Deductible",
    "PolicyAnnualPremium",
    "UmbrellaLimit",
    "VehicleMake",
    "VehicleModel",
    "VehicleYOM",
    "TypeOfIncident",
    "TypeOfCollission",  # Spelled this way in the CSV
    "IncidentState",
    "NumberOfVehicles",
]

X = data[feature_columns].copy()
y = data["ReportedFraud"].map({"N": 0, "Y": 1})
incident_dates = pd.to_datetime(data["DateOfIncident"], format="%Y-%m-%d")

print("Feature table shape:", X.shape)
print("Fraud rate:", f"{y.mean():.1%}")

Feature table shape: (28836, 13)
Fraud rate: 27.0%


In [5]:
validation_start = pd.Timestamp("2015-02-07")
test_start = pd.Timestamp("2015-02-18")

train_mask = incident_dates < validation_start
val_mask = (incident_dates >= validation_start) & (incident_dates < test_start)
test_mask = incident_dates >= test_start

X_train, y_train = X.loc[train_mask].copy(), y.loc[train_mask].copy()
X_val, y_val = X.loc[val_mask].copy(), y.loc[val_mask].copy()
X_test, y_test = X.loc[test_mask].copy(), y.loc[test_mask].copy()

print(f"Training:   {len(X_train):,} claims")
print(f"Validation: {len(X_val):,} claims")
print(f"Test:       {len(X_test):,} claims")

Training:   17,435 claims
Validation: 5,905 claims
Test:       5,496 claims


In [6]:
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_cols = [
    "CustomerLoyaltyPeriod",
    "Policy_Deductible",
    "PolicyAnnualPremium",
    "UmbrellaLimit",
    "VehicleYOM",
    "NumberOfVehicles",
]

categorical_cols = [
    "InsurancePolicyState",
    "Policy_CombinedSingleLimit",
    "VehicleMake",
    "VehicleModel",
    "TypeOfIncident",
    "TypeOfCollission",
    "IncidentState",
]

def prepare_features(frame):
    frame = frame.copy()
    for column in numeric_cols:
        frame[column] = pd.to_numeric(frame[column], errors="coerce")
    frame[categorical_cols] = (
        frame[categorical_cols].astype(object).replace(["?", "???"], np.nan)
    )
    return frame

numeric_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("scale", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
    ("encode", OneHotEncoder(handle_unknown="ignore")),
])

preprocessing = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols),
])

baseline = Pipeline([
    ("preprocess", preprocessing),
    ("model", LogisticRegression(max_iter=1000)),
])

baseline.fit(prepare_features(X_train), y_train)
val_scores = baseline.predict_proba(prepare_features(X_val))[:, 1]

print(f"Validation fraud rate:        {y_val.mean():.3f}")
print(f"Validation average precision: {average_precision_score(y_val, val_scores):.3f}")
print(f"Validation ROC-AUC:           {roc_auc_score(y_val, val_scores):.3f}")

Validation fraud rate:        0.276
Validation average precision: 0.395
Validation ROC-AUC:           0.668


In [7]:
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier

random_forest = Pipeline([
    ("preprocess", clone(preprocessing)),
    ("model", RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1,
    )),
])

random_forest.fit(prepare_features(X_train), y_train)
rf_val_scores = random_forest.predict_proba(prepare_features(X_val))[:, 1]

validation_results = pd.DataFrame([
    {
        "Model": name,
        "Average precision": average_precision_score(y_val, scores),
        "ROC-AUC": roc_auc_score(y_val, scores),
    }
    for name, scores in [
        ("Logistic Regression", val_scores),
        ("Random Forest", rf_val_scores),
    ]
]).sort_values("Average precision", ascending=False)

print(validation_results.to_string(index=False, float_format="{:.3f}".format))

              Model  Average precision  ROC-AUC
      Random Forest              0.614    0.780
Logistic Regression              0.395    0.668


In [8]:
review_capacity = 0.20
max_reviews = int(review_capacity * len(y_val))

# Count how many claims would be flagged at each distinct score cutoff.
queue_sizes = (
    pd.Series(rf_val_scores)
    .value_counts()
    .sort_index(ascending=False)
    .cumsum()
)
eligible_thresholds = queue_sizes[queue_sizes <= max_reviews]

review_threshold = (
    float(eligible_thresholds.index[-1])
    if not eligible_thresholds.empty
    else float(np.nextafter(rf_val_scores.max(), np.inf))
)

flagged = rf_val_scores >= review_threshold
actual_fraud = y_val.to_numpy() == 1
fraud_caught = (flagged & actual_fraud).sum()

print(f"Threshold: {review_threshold:.4f}")
print(f"Claims flagged: {flagged.sum():,}/{len(flagged):,} ({flagged.mean():.1%})")
print(f"Fraud caught: {fraud_caught:,}/{actual_fraud.sum():,} "
      f"({fraud_caught / actual_fraud.sum():.1%} recall)")
print(f"Fraud among flagged claims: {fraud_caught / max(flagged.sum(), 1):.1%}")
print(f"Fraud cases missed: {actual_fraud.sum() - fraud_caught:,}")

Threshold: 0.4050
Claims flagged: 1,181/5,905 (20.0%)
Fraud caught: 750/1,629 (46.0% recall)
Fraud among flagged claims: 63.5%
Fraud cases missed: 879


In [9]:
rf_test_scores = random_forest.predict_proba(prepare_features(X_test))[:, 1]
test_flagged = rf_test_scores >= review_threshold
test_fraud = y_test.to_numpy() == 1

n_flagged = int(test_flagged.sum())
n_fraud = int(test_fraud.sum())
n_caught = int((test_flagged & test_fraud).sum())

print(f"Claims flagged: {n_flagged:,}/{len(y_test):,} ({test_flagged.mean():.1%})")
print(f"Fraud caught: {n_caught:,}/{n_fraud:,} ({n_caught / n_fraud:.1%} recall)")
print(f"Fraud among flagged claims: {n_caught / max(n_flagged, 1):.1%}")
print(f"Fraud cases missed: {n_fraud - n_caught:,}")
print(f"Test average precision: {average_precision_score(y_test, rf_test_scores):.3f}")
print(f"Test ROC-AUC: {roc_auc_score(y_test, rf_test_scores):.3f}")

Claims flagged: 1,070/5,496 (19.5%)
Fraud caught: 346/1,141 (30.3% recall)
Fraud among flagged claims: 32.3%
Fraud cases missed: 795
Test average precision: 0.340
Test ROC-AUC: 0.672


In [10]:
validation_fraud = y_val.to_numpy() == 1
queue_sizes = (
    pd.Series(rf_val_scores)
    .value_counts()
    .sort_index(ascending=False)
    .cumsum()
)

scenarios = []

for capacity in (0.10, 0.20, 0.30):
    max_reviews = int(capacity * len(y_val))
    eligible = queue_sizes[queue_sizes <= max_reviews]
    threshold = (
        float(eligible.index[-1])
        if not eligible.empty
        else float(np.nextafter(rf_val_scores.max(), np.inf))
    )

    referrals = rf_val_scores >= threshold
    n_referred = int(referrals.sum())
    n_caught = int((referrals & validation_fraud).sum())

    scenarios.append({
        "Review capacity": f"{capacity:.0%}",
        "Score cutoff": f"{threshold:.4f}",
        "Claims referred": n_referred,
        "Fraud cases caught": n_caught,
        "Fraud recall": f"{n_caught / validation_fraud.sum():.1%}",
        "Referral precision": f"{n_caught / n_referred:.1%}" if n_referred else "N/A",
    })

validation_scenarios = pd.DataFrame(scenarios)
display(validation_scenarios)

,Review capacity,Score cutoff,Claims referred,Fraud cases caught,Fraud recall,Referral precision
0,10%,0.4810,590,443,27.2%,75.1%
1,20%,0.4050,1181,750,46.0%,63.5%
2,30%,0.3546,1771,963,59.1%,54.4%


In [11]:
from pathlib import Path

scenario_file = Path("validation_scenarios.csv")
validation_scenarios.to_csv(scenario_file, index=False)
print(scenario_file.resolve())

C:\Users\Younis\Desktop\Data Science\archive 2\validation_scenarios.csv
